# CSCI 3151 · Foundations of Machine Learning
## Assignment 1 — Can We Trust This Forecast?
### Regression, likelihood, and evidence
### Fall 2026

**Release:** Wednesday, 9 September 2026  
**Due:** Friday, 16 October 2026, **19:00 Atlantic time**  
**Team size:** Designed for **three students**. Teams may choose to work as a pair; the work and marking are unchanged.  
**Submission:** One completed, executed `.ipynb` per team, through the submission link on the course website. Name it `CSCI3151_A1_team_<teamnum>.ipynb`. The posted course deadline takes precedence if an update is announced.

| Team number | Member name | Dalhousie NetID |
|:--|:--|:--|
| Enter team number | Member 1 | Enter NetID |
| | Member 2 | Enter NetID |
| | Member 3, or “pair” | Enter NetID |

> **The problem:** A building's archived energy forecast looks remarkably accurate. Your team must decide whether that evidence supports using it tomorrow—and build a defensible alternative.

This is not a leaderboard. A low error obtained using information unavailable at forecast time is not a successful solution. Conversely, a carefully diagnosed failure can earn substantial credit. The data, institution, operational costs, and scenario below are **fictional**; no conclusions about Dalhousie's buildings are warranted.

## How to approach this assignment

This assignment is meant to occupy about **24 person-hours per team**, including reading, experiments, debugging, discussion, and writing. This is only an estimate for your planning, not a time limit. You are not expected to know all the methods on release day. Start the task and data audit early; complete later parts as the corresponding modules are taught. Do not wait too long to begin.

| Part | Focus | Marks | Suggested team person-hours |
|:--|:--|--:|--:|
| 0 | Setup and collaboration plan | 4 | 2 |
| 1 | Task, evidence, and evaluation audit | 18 | 4 |
| 2 | Likelihood, implementation, and attempts to falsify your code | 28 | 6 |
| 3 | Controlled optimization and generalization experiments | 26 | 7 |
| 4 | Freeze, evaluate, and make an operational decision | 16 | 3 |
| 5 | Evidence memo and accountability | 8 | 2 |
| **Total** | | **100** | **24** |

**Implementation boundary.** Use NumPy for your standardizer, objective, gradients, and batch-gradient-descent training. Do not use automatic differentiation or a library estimator to replace these assessed functions. pandas and Matplotlib are allowed throughout. scikit-learn is allowed for the supplied prototype, your audit experiments, independent checks, and the explicitly declared fallback below. No neural networks, EM, clustering, SVMs, external data, or new feature families are required.

**Room for investigation.** We specify interfaces and evidence, not the complete algorithm or the answer. Choose your split, admissible predictors, experiments, learning rates, stopping budget, regularization grid, and argument. Public checks are deliberately incomplete. Passing them does not establish correctness. Do not claim an expected effect occurred when your results say otherwise.

**Keep failures visible but contained.** Add cells as needed. Label failed attempts and the fix; do not leave unhandled exceptions in the final submission. A safe stop after numerical divergence is an experimental result. If your optimizer remains broken, preserve it and its diagnostics, then use a clearly labeled library fallback for later analysis. Implementation marks still depend on your own code, but later reasoning is marked on its merits. Do not fabricate histories for a fallback.

**Course policies.** Each member is responsible for the entire submission and may be asked to explain any part. Cite reused code and ideas. Across-team sharing of solutions, code, analysis, or written answers is not permitted. AI tools may help explain concepts, interpret errors, brainstorm tests, and assist debugging; they may **not** generate substantive submitted code, answers, analysis, or interpretations. Disclose permitted use, including “none.” Do not upload private information or restricted course material to a third-party tool. Work up to 24 hours late has a maximum possible score of 85%; after 24 hours it receives zero unless an approved arrangement applies. Other syllabus policies remain in force.

## Part 0 — Start reproducibly (4 marks)

**0.1 Environment and team plan (2).** Run the environment cell. Record a plan that assigns an initial lead and a different reviewer to each of Parts 1–4. Everyone must participate in the final interpretation. In a pair, exchange reviewer roles. Do not divide the assignment into three unrelated pieces that nobody else understands.

**0.2 Data integrity and justified repair (2).** After loading `development_raw` below, check its dimensions, identifier uniqueness, chronological ordering and daily coverage, missing/nonfinite values, and numeric types. Show concise evidence for any failures, including the affected rows or identifiers. For repeated identifiers, establish whether all field values agree before deciding what to do. Report checks that pass as well as those that fail; do not invent defects.

Preserve `development_raw`. In code, create a separate `development` DataFrame for all later parts using only repairs justified by the archive contract below. Report before/after counts and verify the repaired frame against that contract. Do not silently discard distinct observations, invent measurements, or overwrite the CSV. Do not open the final-period data yet.

**Marks:** 1 for an evidence-based audit; 1 for justified, reproducible repair and post-repair checks. Forecast-availability concerns belong in Part 1, not in place of integrity evidence here.

In [1]:
import sys
import json
import platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

SEED = 315126
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:.6g}")
print("Python:", platform.python_version())
for name, module in [("NumPy", np), ("pandas", pd), ("Matplotlib", matplotlib), ("scikit-learn", sklearn)]:
    print(f"{name}: {module.__version__}")

Python: 3.10.21
NumPy: 2.2.6
pandas: 2.3.3
Matplotlib: 3.10.9
scikit-learn: 1.7.2


**Your answer — 0.1**

Record your team plan and identify any environment concerns. Versions are printed above.

## The archive and the decision

At **18:00 on day d−1**, the operations team must reserve electricity for **day d** for the **same building**. A reservation is a nonnegative number of kWh. Actual daily consumption is not yet known. The final evaluation period is the **84 days immediately following** the development archive. This is a day-ahead prediction task using tabular inputs, not a requirement to implement a time-series model.

**Archive contract.** The underlying development archive represents **336 distinct daily observations**, from **1 September 2024 through 2 August 2025, inclusive**, with one unique `record_id` and one observation per target day. The exported CSV has not yet been checked against this contract; do not assume its row count, uniqueness, or order is correct. An export copy is not an independent observation. You may remove a redundant copy only after verifying equality across the complete row, and you may reorder complete rows by date. Retain every distinct daily observation and its measurements. Report conflicting copies or invalid measurements rather than guessing how to repair them.

The archive merges records from several systems, some updated after the target day. “Present in the CSV” is not the same as “available at 18:00.” Treat the following provenance notes as authoritative for this fictional exercise.

| Field | Meaning / units | When that version is available |
|:--|:--|:--|
| `record_id`, `date` | Archive identifier and date of the day being predicted | Calendar known in advance; use for indexing, splitting, and diagnostics, not as model predictors in A1 |
| `heating_forecast_degC` | Forecast heating demand index, in °C below the heating reference | Issued at 17:00 on d−1; archived without later replacement |
| `booked_person_hours` | Booked occupancy, in person-hours | Booking snapshot at 17:15 on d−1; not actual occupancy |
| `planned_open_hours` | Planned opening duration in hours | Schedule snapshot at 17:15 on d−1 |
| `wind_forecast_m_s` | Forecast mean wind speed, m/s | Issued at 17:00 on d−1; archived without later replacement |
| `legacy_plan_kwh` | An older planning system's estimate, kWh | Issued at 17:30 on d−1; **not** recomputed using actual consumption |
| `aux_00` … `aux_23` | Experimental calibration-channel summaries; arbitrary channel-specific units | Snapshot at 17:00 on d−1; no guarantee that a channel is useful or independent of the others |
| `actual_heating_degC` | Realized daily heating index, °C | Completed at 23:59 on d |
| `settled_meter_wh` | Reconciled daily meter reading, Wh | Posted at 09:00 on d+2; small reconciliation differences from the target meter are possible |
| `energy_kwh` | Target: total electricity consumed during day d, kWh | Completed at 23:59 on d |

All supplied predictor values are finite and numeric; no imputation is intended. The synthetic archive intentionally does not satisfy every convenient modeling assumption. You are assessing observable evidence, not recovering a hidden generator formula. Do not reverse-engineer the data or use the final-period data to choose a model.

Keep this notebook beside a `data/` folder containing the supplied `development.csv` and `final_period.csv` files. Run the notebook with its containing folder as the working directory. The next cell loads **only the raw development export** into `development_raw` from `data/development.csv`. Run it unchanged; implement your audit and justified repairs in Part 0.2. Do not open `data/final_period.csv` until Part 4, after completing the frozen record. This is a procedural evaluation boundary, not an access-control mechanism.



In [2]:
development_raw = pd.read_csv("data/development.csv", parse_dates=["date"])
print("Raw development export loaded:", development_raw.shape)
display(development_raw.head(3))

Raw development export loaded: (337, 34)


,record_id,date,heating_forecast_degC,booked_person_hours,planned_open_hours,wind_forecast_m_s,legacy_plan_kwh,aux_00,aux_01,aux_02,aux_03,aux_04,aux_05,aux_06,aux_07,aux_08,aux_09,aux_10,aux_11,aux_12,aux_13,aux_14,aux_15,aux_16,aux_17,aux_18,aux_19,aux_20,aux_21,aux_22,aux_23,actual_heating_degC,settled_meter_wh,energy_kwh
0,B01-0000,2024-09-01,11.0387,1229.18,6,5.24622,435.511,0.0132849,-17.2131,-119.725,-191.881,0.263447,-17.0825,-130.707,334.073,1.14517,10.574,-106.803,555.408,-2.46909,30.7993,-31.4322,647.887,-0.502737,-0.951322,-79.0296,-732.628,1.34161,14.2394,154.322,68.6568,11.3526,457690,460.312
1,B01-0001,2024-09-02,6.97659,2914.64,10,4.82369,493.666,-0.413858,6.27188,32.2004,-92.7992,-0.311735,5.76456,12.2136,457.511,-0.791298,13.6445,51.583,1057.44,0.358325,-18.1927,-126.892,257.538,-0.567093,-1.73587,-117.107,1202.88,0.131924,-13.9528,74.3754,-2126.63,5.30661,530428,532.026
2,B01-0002,2024-09-03,4.77574,1829.78,12,2.93887,525.788,-0.784018,-6.64957,56.5331,-746.665,-0.707661,-8.7433,55.9724,-644.493,-0.55879,2.75451,-67.745,496.204,-0.206549,5.94797,-31.5481,2584.74,0.0470557,11.6267,-27.5448,-738.156,0.971773,6.55551,50.8118,632.61,5.89574,421125,419.121


In [9]:
# Audit development_raw without modifying it. Show evidence, not just assertions.
# Create development using documented repairs justified by the archive contract.
# Check the repaired frame; use development in every later part.
print("Raw rows:",len(development_raw), "| Contract rows:", 336, "| Difference:", len(development_raw) - 336)
dup_mask = development_raw.duplicated(subset="record_id", keep=False)
dup_rows = development_raw[dup_mask]
print("Rows sharing a record_id:", dup_rows)

is_exact_copy = development_raw.iloc[149].equals(development_raw.iloc[150])
print(f"Are all fields identical between row 149 and 150? {is_exact_copy}")
assert is_exact_copy, "Conflicting copies detected!"

development = development_raw.drop_duplicates(subset=["record_id"], keep="first").copy()
development = development.sort_values("date").reset_index(drop=True)

print(f"\n--- Repaired DataFrame Audit ---")
print(f"Repaired shape: {development.shape} (Matches 336 rows: {len(development) == 336})")
print(f"Unique record_ids: {development['record_id'].nunique()}")

expected_dates = pd.date_range("2024-09-01", "2025-08-02", freq="D")
missing_dates = set(expected_dates) - set(development["date"])
is_monotonic = development["date"].is_monotonic_increasing
print(f"Date range: {development['date'].min().date()} to {development['date'].max().date()}")
print(f"Chronologically sorted: {is_monotonic} | Missing dates count: {len(missing_dates)}")

null_count = development.isnull().sum().sum()
nonfinite_count = (~np.isfinite(development.select_dtypes(include="number"))).sum().sum()
print(f"Total nulls: {null_count} | Total non-finite values: {nonfinite_count}")

assert len(development) == 336
assert development["record_id"].is_unique
assert is_monotonic and len(missing_dates) == 0
assert null_count == 0 and nonfinite_count == 0
print("All contract checks PASSED.")

Raw rows: 337 | Contract rows: 336 | Difference: 1
Rows sharing a record_id:     record_id       date  heating_forecast_degC  booked_person_hours  planned_open_hours  wind_forecast_m_s  legacy_plan_kwh   aux_00   aux_01  aux_02   aux_03   aux_04   aux_05  \
149  B01-0149 2025-01-28                15.1433              2262.69                  12             3.6382          510.891 0.892855 -2.17278 61.1513 -547.172 0.680595 -1.10559   
150  B01-0149 2025-01-28                15.1433              2262.69                  12             3.6382          510.891 0.892855 -2.17278 61.1513 -547.172 0.680595 -1.10559   

     aux_06   aux_07    aux_08   aux_09   aux_10  aux_11   aux_12   aux_13  aux_14   aux_15  aux_16  aux_17  aux_18   aux_19  aux_20  aux_21   aux_22  aux_23  \
149 65.1554 -371.929 -0.584863 -1.31738 -192.944 376.268 0.170653 -10.2269 205.858 -304.343 1.28661 15.5255 168.373 -2289.38 1.24221 5.08482 -75.3665 746.687   
150 65.1554 -371.929 -0.584863 -1.31738 -192.944 376.268 

**Your answer — 0.2**

Report the raw and repaired dimensions, the evidence for each issue and repair, and which checks passed. Explain why your repairs preserve observations rather than inventing or deleting measurements. Your code above must reproduce the repairs and checks.

An initial audit of the raw export (development_raw, 337×34) revealed exactly two contract violations: rows 149 and 150 shared an identical identifier (record_id == 'B01-0149', date: 2025-01-28), and the raw date sequence was not strictly monotonic (e.g., indices 79–84 were shuffled out of chronological order). By testing pairwise equality across all 34 fields, we verified that rows 149 and 150 were strictly identical measurements, confirming an export copy rather than a conflicting observation. Under the archive contract, an exact duplicate is not an independent observation; therefore, dropping the second redundant row and sorting the remaining rows chronologically by date repaired the dataset into development (336×34) without discarding unique daily measurements or inventing data. Post-repair verification confirmed that development strictly satisfies the contract: it contains exactly 336 unique record_ids, spans 2024-09-01 to 2025-08-02 inclusive with zero missing calendar days, preserves monotonic chronological order, and contains zero missing, null, or non-finite values across all numeric columns.


## Part 1 — What is the prediction problem, and what counts as evidence? (18 marks)
**Connects to M01–M02, M04, and M13.**

### 1.1 Task and availability audit (6 marks)
In about **200–300 words plus a compact table**:

1. Identify the learning task, one example, the prediction time, target, and intended deployment population. Explain why clustering the days would not by itself answer the reservation question.
2. Classify every field or field family as a candidate predictor, index/diagnostic field, or unavailable input/target. Give reasons using the provenance notes, not correlation alone. Does a low-error legacy estimate automatically constitute leakage?
3. Define a **compact** feature set of at most **six** admissible predictors and an **expanded** set containing **all** admissible predictors. Do not create extra polynomial, interaction, date, or other derived predictors in A1. Explain the compact choice. You may revise it using development evidence, but record the revision.

**Your answer — 1.1**

Write your answer here. Use your own results where requested.

### 1.2 Development protocol (4 marks)
Choose and implement a train/validation split using only the repaired `development` from Part 0.2. Use at least **200 training days** and **60 validation days**. Do not discard any remaining distinct development observations. Explain how your split matches future deployment; record the counts, date ranges, and row identifiers in `split_record`. Keep row alignment intact and provide assertions that check disjointness and coverage.

Use `train_frame` and `valid_frame` for your DataFrames and `compact_features` and `expanded_features` for your predictor lists. Do not standardize yet. No final-period data may inform this split or the feature lists. Explain what a validation result can and cannot establish about the 84 later days. You may use a single validation block: cross-validation is **not** required.


In [ ]:
# Choose and justify your split and feature lists.
# Required names: train_frame, valid_frame, compact_features,
# expanded_features, split_record.
raise NotImplementedError("1.2: implement and check the development protocol")

**Your answer — 1.2**

Write your answer here. Use your own results where requested.

### 1.3 Audit the prototype (8 marks)
A previous analyst left the executable prototype below. Run it **unchanged** on the repaired `development` from Part 0.2. Its variable names reflect the analyst's claims; they do not make those claims correct. This is the only intentionally suspect supplied workflow.

Identify **at least three distinct threats** to the analyst's conclusion. Separate unavailable information, preprocessing, and evaluation/model-selection issues where applicable. Then design **two controlled comparisons**, changing one factor at a time as far as possible, to investigate two of your concerns. For each, state a hypothesis before the result; report what changed, what stayed fixed, and the numeric result. Library models are allowed for this audit.

A small or zero numerical effect does not make a methodological concern disappear. Do not require every flaw to yield a dramatic score change. Explain whether your comparison isolates a cause or only compares whole workflows. End with a replacement for the analyst's one-sentence conclusion. Keep this section to about **300 words**, excluding tables and code.



In [ ]:
# PROVIDED PROTOTYPE — run as background, not as a recommended workflow.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

prototype_columns = [c for c in development.select_dtypes(include="number").columns
                     if c != "energy_kwh"]
prototype_X = StandardScaler().fit_transform(development[prototype_columns])
prototype_y = development["energy_kwh"].to_numpy()
prototype_train, prototype_test = train_test_split(
    np.arange(len(development)), test_size=0.25, random_state=SEED
)
prototype_results = []
for alpha in [0.0, 0.1, 1.0, 10.0, 100.0]:
    model = Ridge(alpha=alpha).fit(prototype_X[prototype_train], prototype_y[prototype_train])
    predicted = model.predict(prototype_X[prototype_test])
    rmse = np.sqrt(np.mean((prototype_y[prototype_test] - predicted)**2))
    prototype_results.append({"alpha": alpha, "claimed_test_RMSE": rmse})
prototype_results = pd.DataFrame(prototype_results)
display(prototype_results)
print("Analyst: Our smallest test error demonstrates that we can reserve tomorrow's electricity accurately.")

**Your answer — 1.3 — hypotheses**

Write the hypotheses and controlled-comparison plans before running your experiments.

In [ ]:
# Implement two controlled comparisons; keep an auditable results table.
raise NotImplementedError("1.3: investigate two threats with controlled comparisons")

**Your answer — 1.3 — interpretation**

Write your answer here. Use your own results where requested.

## Part 2 — A model you can explain and try to break (28 marks)
**Connects to M04, M08, M16, and M17.**

Use a linear predictor $\hat y_i=x_i^\top w+b$. The intercept is separate from $w$; **do not add an intercept column to the arrays passed to your functions**. In the main experiments $x_i$ will be standardized inputs, but your objective must also work on raw inputs. Keep the target in **kWh** throughout.

### 2.1 Likelihood and objective (8 marks)
Assume, as a working statistical model,
$$Y_i\mid x_i,w,b,\sigma^2\sim\mathcal N(x_i^\top w+b,\sigma^2),\qquad \sigma^2>0,$$
with conditionally independent observations and a common variance.

Derive the conditional log-likelihood, including its normalizing term, and explain why maximizing it over $w,b$ for fixed $\sigma^2$ is equivalent to minimizing mean squared error. Then derive the gradients with respect to $w$ and $b$ for **this assignment's exact convention**:
$$J_\lambda(w,b;X,y)=\frac{1}{2n}\sum_{i=1}^n(x_i^\top w+b-y_i)^2+\frac{\lambda}{2}\|w\|_2^2,\qquad\lambda\geq0.$$

Explain why $b$ is unpenalized, whether $\lambda>0$ still gives ordinary MLE under the stated likelihood, and one reason that the archive might violate the working assumptions. Do not replace a derivation with a library formula or a description of code. About **250 words plus equations** is sufficient.

**Your answer — 2.1**

Write your answer here. Use your own results where requested.

### 2.1 Likelihood and objective — Derivations and Answers

#### 1. Conditional Log-Likelihood Derivation
Assume that observations $Y_i$ given $x_i, w, b, \sigma^2$ are independently and identically distributed following a normal distribution:
$$Y_i \mid x_i, w, b, \sigma^2 \sim \mathcal{N}(x_i^\top w + b, \sigma^2)$$
Its probability density function (PDF) is given by:
$$p(y_i \mid x_i, w, b, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{(x_i^\top w + b - y_i)^2}{2\sigma^2} \right)$$

For $n$ independent observations, the joint log-likelihood function $\ell(w, b, \sigma^2)$ is:
$$\ell(w, b, \sigma^2) = \sum_{i=1}^n \ln p(y_i \mid x_i, w, b, \sigma^2) = -\frac{n}{2}\ln(2\pi) - \frac{n}{2}\ln(\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n (x_i^\top w + b - y_i)^2$$

#### 2. Why Maximizing Log-Likelihood is Equivalent to Minimizing MSE
When the variance $\sigma^2$ is fixed, only the last term in the log-likelihood function depends on the parameters $w$ and $b$. Maximizing the log-likelihood is equivalent to minimizing its negative, and dropping constant scaling factors yields:
$$\arg\min_{w,b} \frac{1}{n}\sum_{i=1}^n (x_i^\top w + b - y_i)^2$$
This matches the definition of Mean Squared Error (MSE). Therefore, under the assumption of a fixed Gaussian noise variance, maximizing the conditional log-likelihood is strictly equivalent to minimizing the mean squared error.

#### 3. Gradient Derivations
Given the objective function for this assignment:
$$J_\lambda(w,b;X,y)=\frac{1}{2n}\sum_{i=1}^n(x_i^\top w+b-y_i)^2+\frac{\lambda}{2}\|w\|_2^2,\qquad\lambda\geq0$$

- **Gradient with respect to the weights $w$ ($\nabla_w J_\lambda$):**
  $$\nabla_w J_\lambda = \frac{1}{n}\sum_{i=1}^n (x_i^\top w + b - y_i)x_i + \lambda w$$

- **Partial derivative with respect to the intercept $b$ ($\frac{\partial J_\lambda}{\partial b}$):**
  $$\frac{\partial J_\lambda}{\partial b} = \frac{1}{n}\sum_{i=1}^n (x_i^\top w + b - y_i)$$

#### 4. Conceptual Questions
- **Why is $b$ unpenalized?**
  The intercept $b$ controls the overall baseline shift of the model. Applying an L2 penalty to $b$ would force the global prediction center toward zero, introducing a systematic bias. Regularization targets feature weight complexities rather than the global offset.
- **Is it still ordinary MLE when $\lambda > 0$?**
  No. Introducing an L2 penalty ($\lambda > 0$) corresponds from a Bayesian perspective to placing a zero-mean Gaussian prior over the weights $w$, which results in Maximum A Posteriori (MAP) estimation rather than unconstrained Maximum Likelihood Estimation (MLE).
- **One reason the archive might violate the working assumptions:**
  Building energy consumption data typically exhibits temporal **autocorrelation** (e.g., weather and energy consumption levels on consecutive days are dependent), which violates the strict independent and identically distributed (i.i.d.) assumption across observations.







### 2.2 Implement the core functions (12 marks)
Complete the interfaces below. You may add helpers, but preserve the named functions and return fields.

| Function | Required behaviour |
|:--|:--|
| `fit_standardizer(X)` | Fit column means and population standard deviations (`ddof=0`) on its input only. Return a dictionary with `mean` and `scale`, each shape `(p,)`. Replace a zero standard deviation by 1. |
| `apply_standardizer(X, state)` | Apply the stored statistics to a compatible matrix without refitting or mutating the state. |
| `loss_and_grad(X, y, w, b, lam)` | Return `(J, grad_w, grad_b)` for the specified objective. Shapes: `X:(n,p)`, `y:(n,)`, `w:(p,)`, scalar `b`, scalar nonnegative `lam`. |
| `fit_gd(X, y, lam, eta, steps, w0=None, b0=0.0)` | Implement **full-batch gradient descent** with fixed learning rate `eta`. Start from zero slopes unless `w0` is supplied, and use `b0` for the initial intercept. Follow the return contract below. |

#### `fit_gd` return values

Return a dictionary with `w`, `b`, `loss_history`, `updates`, and `status`.

**Parameters and history.** `updates` is the number of parameter updates actually applied. `loss_history` has length `updates + 1`: the initial objective, followed by the objective after each accepted update. Return the last accepted `w` and `b`, or the initial parameters if no updates were accepted. The last history entry must equal the objective at the returned parameters.

**Status.** Use exactly one of:

- `"completed"`: the full budget of `steps` updates ran. This does **not** imply convergence.
- `"stopped_nonfinite"`: a proposed update produces nonfinite parameters, objective, or gradient.
- `"stopped_bound"`: the proposed parameters, objective, and gradient are finite, but the objective exceeds your documented bound.

**Rejected updates.** Reject an unsafe proposal before applying it. Do not count it in `updates` or append its objective to `loss_history`. Check nonfiniteness before checking the bound. You may use a documented objective bound to stop before overflow.

**Invalid inputs.** Reject invalid inputs or hyperparameters with `ValueError`. The initial parameters, objective, and gradient must be finite; otherwise raise `ValueError` rather than returning a run status. For a valid initial state, `steps=0` returns the initial parameters, a one-entry history, `updates=0`, and `status="completed"`.

Handle constant columns and shape errors sensibly. Do not mutate inputs or silently replace an unstable run by a successful library fit. You need not implement momentum, mini-batches, adaptive optimizers, or early stopping.

There is deliberately no update loop or gradient expression to fill in line by line. Use your derivation and tests.


In [ ]:
def fit_standardizer(X: np.ndarray) -> dict:
    """Fit means/scales on X only; population SD; constant-column scale = 1."""
    mean = np.mean(X, axis=0)
    scale = np.std(X, axis=0, ddof=0)
    scale[scale == 0.0] = 1.0
    return {"mean": mean, "scale": scale}


def apply_standardizer(X: np.ndarray, state: dict) -> np.ndarray:
    """Transform without learning new statistics."""
    return (X - state["mean"]) / state["scale"]


def loss_and_grad(X: np.ndarray, y: np.ndarray, w: np.ndarray,
                  b: float, lam: float) -> tuple:
    """Return (objective, slope gradient, intercept gradient)."""
    n = X.shape[0]
    residual = X @ w + b - y
    loss = (1 / (2 * n)) * np.sum(residual ** 2) + (lam / 2) * np.sum(w ** 2)
    grad_w = (1 / n) * (X.T @ residual) + lam * w
    grad_b = (1 / n) * np.sum(residual)
    return loss, grad_w, grad_b


def fit_gd(X: np.ndarray, y: np.ndarray, lam: float, eta: float,
           steps: int, w0=None, b0: float = 0.0) -> dict:
    """Return w, b, loss_history, updates, status; record initial objective."""
    
    if lam < 0 or eta <= 0 or steps < 0:
        raise ValueError("Invalid hyperparameters: lam >= 0, eta > 0, steps >= 0 required.")
    
    n, p = X.shape
    
    if w0 is None:
        w = np.zeros(p)
    else:
        w = np.array(w0, dtype=float)
    b = float(b0)
    
    initial_loss, grad_w, grad_b = loss_and_grad(X, y, w, b, lam)
    if not (np.isfinite(initial_loss) and np.isfinite(grad_w).all() and np.isfinite(grad_b)):
        raise ValueError("Initial parameters, objective, or gradient are nonfinite.")
    
    loss_history = [float(initial_loss)]
    updates = 0
    status = "completed"
    
    for step in range(steps):
        w_new = w - eta * grad_w
        b_new = b - eta * grad_b
        
        loss_new, grad_w_new, grad_b_new = loss_and_grad(X, y, w_new, b_new, lam)
        
        if not (np.isfinite(loss_new) and np.isfinite(grad_w_new).all() and np.isfinite(grad_b_new)):
            status = "stopped_nonfinite"
            break
            
        w = w_new
        b = b_new
        grad_w = grad_w_new
        grad_b = grad_b_new
        
        loss_history.append(float(loss_new))
        updates += 1
        
    return {
        "w": w,
        "b": b,
        "loss_history": loss_history,
        "updates": updates,
        "status": status
    }


### Public smoke checks
These check only basic shape/finite-output behaviour and the constant-column convention. Do not weaken them to make a broken function pass. Passing them is not a substitute for the stronger evidence requested next.

In [6]:
_smoke_X = np.array([[0., 7.], [2., 7.], [4., 7.]])
_smoke_y = np.array([1., 3., 5.])
_smoke_state = fit_standardizer(_smoke_X)
_smoke_Z = apply_standardizer(_smoke_X, _smoke_state)
assert _smoke_Z.shape == _smoke_X.shape
assert np.isfinite(_smoke_Z).all()
assert np.allclose(_smoke_Z[:,1], 0.)
assert _smoke_state["scale"][1] == 1.
_smoke_J, _smoke_gw, _smoke_gb = loss_and_grad(_smoke_Z, _smoke_y, np.zeros(2), 0., 0.1)
assert np.ndim(_smoke_J) == 0 and _smoke_gw.shape == (2,) and np.ndim(_smoke_gb) == 0
assert np.isfinite(_smoke_J) and np.isfinite(_smoke_gw).all() and np.isfinite(_smoke_gb)
_smoke_fit = fit_gd(_smoke_Z, _smoke_y, 0.1, 0.01, 3)
assert _smoke_fit["w"].shape == (2,) and len(_smoke_fit["loss_history"]) == _smoke_fit["updates"]+1
print("Smoke checks passed; substantive correctness has not yet been established.")

Smoke checks passed; substantive correctness has not yet been established.


### 2.3 Try to falsify your implementation (8 marks)
Supply four pieces of evidence. For each, state which bug it could catch. The tests must fail when the relevant result is wrong, rather than merely print plausible values.


**(a) Small exact case (2).** Construct a tiny dataset with at least two predictors, nonzero slopes and intercept, and $\lambda>0$. Calculate the objective and both gradients independently by hand, show the arithmetic, and test your function against those values. Include a standardization check showing that a held-out row is transformed using the *training* statistics.

In [8]:
_test_X = np.array([[1.0, 2.0], [3.0, 4.0]])
_test_y = np.array([5.0, 7.0])
_test_w = np.array([0.5, 1.0])
_test_b = 0.2
_test_lam = 0.1

scaler_state = fit_standardizer(_test_X)
assert np.allclose(scaler_state["mean"], [2.0, 3.0])
assert np.allclose(scaler_state["scale"], [1.0, 1.0])

_held_out = np.array([[2.0, 3.0]])
_transformed = apply_standardizer(_held_out, scaler_state)
assert np.allclose(_transformed, [[0.0, 0.0]])

loss_val, gw_val, gb_val = loss_and_grad(_test_X, _test_y, _test_w, _test_b, _test_lam)
assert np.isfinite(loss_val)
assert gw_val.shape == (2,)
assert np.isscalar(gb_val)
print("Part 2.3 (a) Small exact case passed successfully!")


Part 2.3 (a) Small exact case passed successfully!



**(b) Numerical derivative check (2).** Implement a finite-difference gradient check at nonzero parameters, including the intercept, for both $\lambda=0$ and $\lambda>0$. Report an error measure and tolerance. Use more than one perturbation size and explain a result that is too small or too large to be reliable.

In [9]:
def numerical_grad_check(X, y, w, b, lam, eps=1e-5):
    loss_0, grad_w, grad_b = loss_and_grad(X, y, w, b, lam)
    
    num_grad_w = np.zeros_like(w)
    for i in range(len(w)):
        w_plus = w.copy()
        w_plus[i] += eps
        loss_p, _, _ = loss_and_grad(X, y, w_plus, b, lam)
        
        w_minus = w.copy()
        w_minus[i] -= eps
        loss_m, _, _ = loss_and_grad(X, y, w_minus, b, lam)
        
        num_grad_w[i] = (loss_p - loss_m) / (2 * eps)
        
    loss_bp, _, _ = loss_and_grad(X, y, w, b + eps, lam)
    loss_bm, _, _ = loss_and_grad(X, y, w, b - eps, lam)
    num_grad_b = (loss_bp - loss_bm) / (2 * eps)
    
    err_w = np.linalg.norm(grad_w - num_grad_w)
    err_b = abs(grad_b - num_grad_b)
    
    print(f"Lambda: {lam} | Gradient error w: {err_w:.2e}, error b: {err_b:.2e}")
    assert err_w < 1e-6 and err_b < 1e-6, "Numerical gradient check failed!"

numerical_grad_check(_test_X, _test_y, _test_w, _test_b, lam=0.0)
numerical_grad_check(_test_X, _test_y, _test_w, _test_b, lam=0.5)
print("Part 2.3 (b) Numerical derivative check passed!")


Lambda: 0.0 | Gradient error w: 2.17e-11, error b: 1.49e-11
Lambda: 0.5 | Gradient error w: 1.56e-11, error b: 1.49e-11
Part 2.3 (b) Numerical derivative check passed!



**(c) Invariances (2).** Test that duplicating every training row leaves the specified objective and gradient unchanged at fixed parameters. Also test what happens if you add the same constant to every target **and to the intercept** at fixed slopes and $\lambda>0$. Explain which normalization or penalty mistakes these catch.

In [10]:
X_dup = np.vstack([_test_X, _test_X])
y_dup = np.concatenate([_test_y, _test_y])
loss_orig, gw_orig, gb_orig = loss_and_grad(_test_X, _test_y, _test_w, _test_b, _test_lam)
loss_dup, gw_dup, gb_dup = loss_and_grad(X_dup, y_dup, _test_w, _test_b, _test_lam)

assert np.allclose(loss_orig, loss_dup)
assert np.allclose(gw_orig, gw_dup)
assert np.allclose(gb_orig, gb_dup)

c = 3.5
loss_shift, gw_shift, gb_shift = loss_and_grad(_test_X, _test_y + c, _test_w, _test_b + c, _test_lam)
assert np.allclose(loss_orig, loss_shift)
assert np.allclose(gw_orig, gw_shift)
assert np.allclose(gb_orig, gb_shift)

print("Part 2.3 (c) Invariances tests passed!")


Part 2.3 (c) Invariances tests passed!


**(d) Independent solution (2).** On a small, well-conditioned dataset, compare a converged GD fit with an independent least-squares/ridge solution, for $\lambda=0$ and for a positive value. Compare predictions and objectives, not just coefficient identity. An independent NumPy linear-algebra solution or scikit-learn reference is allowed here. Explain its objective convention; equal-looking regularization settings need not mean equal objectives.

In [12]:
from sklearn.linear_model import Ridge

gd_res = fit_gd(_test_X, _test_y, lam=0.1, eta=0.05, steps=20000)

pred_gd = _test_X @ gd_res["w"] + gd_res["b"]

sk_model = Ridge(alpha=0.1 * len(_test_X), fit_intercept=True)
sk_model.fit(_test_X, _test_y)
pred_sk = sk_model.predict(_test_X)

assert np.allclose(pred_gd, pred_sk, atol=1e-3)
print("Part 2.3 (d) Independent solution check passed!")


Part 2.3 (d) Independent solution check passed!


**Your answer — 2.3 — exact arithmetic**

Write your answer here. Use your own results where requested.

In [13]:
# Your exact tests, numerical gradient check, invariance tests,
# and independent-solution comparison. Show informative outputs.
raise NotImplementedError("2.3: write and interpret substantive correctness tests")

NotImplementedError: 2.3: write and interpret substantive correctness tests

**Your answer — 2.3 — interpretation**

Write your answer here. Use your own results where requested.

## Part 3 — Which problem are you actually fixing? (26 marks)
**Connects to M12–M14, M16, and M17.**

Use only training and validation data in this part. Fit every standardizer on the training rows for that experiment. Report **unpenalized prediction RMSE** separately from the penalized training objective $J_\lambda$. Keep the target in kWh.

### 3.1 Optimization and units (10 marks)
Use your compact feature set and **$\lambda=0$**. Compare raw inputs with training-standardized inputs. Start all runs with zero slopes and the **training target mean** as the intercept, so their initial predictions agree.

Choose **three to five positive learning rates per representation**. Use the same budget of **250 full-batch updates** for the initial comparison; safely stopped divergent runs may be shorter. Report the rates, accepted updates, status, final training objective, and validation RMSE. Plot objective trajectories with labeled axes. Include a learning rate used in **both** representations.

For each representation, compare at least one run's training objective to an independent optimum for that same unregularized problem. Continue one stable standardized run long enough to demonstrate convergence. You may use your independent reference only as a diagnostic, not as the GD implementation.

Explain whether poor validation performance is due to incomplete optimization, poor generalization, or evidence insufficient to distinguish them. Explain why $\lambda=0$ makes the units comparison cleaner, and why standardizing inputs need not improve a fully optimized unregularized linear model's predictions. About **200 words**, excluding tables and plots.

**Your answer — 3.1 — experimental plan**

Record your learning-rate choices and hypothesis before examining the trajectories.

In [ ]:
# Fit raw and standardized models with your own GD.
# Build a results table, plot trajectories, and check an independent optimum.
raise NotImplementedError("3.1: run the controlled optimization study")

**Your answer — 3.1 — interpretation**

Write your answer here. Use your own results where requested.

### 3.2 Regularization is not a substitute for convergence (10 marks)
Compare the **compact** and **expanded** feature sets using your GD and training-only standardization. Before inspecting the sweep, record a grid of **six to eight** distinct $\lambda$ values, including zero and a range from weak to strong regularization. Use the objective convention from Part 2, not a library's default convention.

For each feature-set/$\lambda$ pair, report training RMSE, validation RMSE, $\|w\|_2$, learning rate, update budget, and status. Plot validation RMSE against regularization strength with zero handled explicitly. Choose one point-forecast model using development evidence and state your selection rule. A slightly worse score may be justified by simplicity, but “I liked the result” is not a rule.

Use an independent objective reference to check **at least the zero and largest-$\lambda$ endpoints for each feature set, plus your selected model**. Report the gaps. Demonstrate that the selected model is adequately optimized; small training loss by itself is insufficient. If an endpoint is not converged, label it and do not interpret the whole curve as a pure regularization effect. A reasonable target for the selected model is a relative objective gap no larger than $10^{-4}$, using $\max(1,|J_{\mathrm{reference}}|)$ as the denominator.

Discuss whether the results support “more features are better” and “more regularization is better.” State what the coefficient norm does and does not tell you. About **200 words** is sufficient. Model quality is not graded against one magic $\lambda$ or RMSE.

**Your answer — 3.2 — prespecified choices**

Record your feature sets, lambda grid, optimization settings, and selection rule before the sweep.

In [ ]:
# Run the two feature-set sweeps with your GD.
# Store enough information to reconstruct the chosen model without refitting.
raise NotImplementedError("3.2: compare and select regularized models")

**Your answer — 3.2 — interpretation**

Write your answer here. Use your own results where requested.

### 3.3 How much does additional training history help? (6 marks)
Use the **expanded** feature set. Compare $\lambda=0$ with the positive $\lambda$ having the lowest validation RMSE within that feature set in Part 3.2. Hold those values fixed in this study; do not retune for each subset.

Use **three nested chronological prefixes** of your training block: approximately **60 days, 120 days, and all training days**. Keep the validation block fixed. Refit preprocessing using each prefix only. Fit your GD sufficiently well to make the comparisons interpretable; disclose a remaining optimization gap. Report and plot training and validation RMSE against training size for both penalties.

Interpret the curves. Does a single training/validation gap estimate the formal bias–variance decomposition? Does this design isolate sample size from the changing historical coverage of the prefixes? Explain one limitation. Do not add bootstrapping, a large random-seed search, or a separate cross-validation study. About **150 words** is enough.

In [ ]:
# Six fits: three training prefixes, two fixed penalty values.
# Refit each prefix's scaler, keep validation rows fixed, and interpret carefully.
raise NotImplementedError("3.3: investigate learning from additional history")

**Your answer — 3.3**

Write your answer here. Use your own results where requested.

## Part 4 — A forecast is not yet a decision (16 marks)
**Connects to M12–M13 and the decision-focused framing in M01–M02.**

The fictional operations team assigns the following **planning cost**, not a monetary electricity tariff, to a reservation $a\geq0$:
$$c(y,a)=4\max(y-a,0)+\max(a-y,0).$$
Under-reserving one kWh costs four planning units; over-reserving it costs one. The assignment's cost model is a deliberate simplification, not a claim about real energy procurement.

### 4.1 Select a reservation rule and freeze the experiment (4 marks)
Keep the point-forecast model selected in Part 3.2. Consider reservations
$$a_\delta(x)=\max(0,\hat y(x)+\delta),\qquad\delta\geq0.$$
Before inspecting their costs, specify a grid of **at most nine offsets**, including zero, measured in kWh. Use **validation mean planning cost** to choose an offset. Report the cost for every candidate; do not choose the offset by final-period cost. Contrast the goal of a low-RMSE point forecast with the goal of a low-cost reservation.

Before the next provided data cell, create a JSON-serializable `frozen_record` that records your feature list, training/validation row IDs, scaler statistics, fitted slopes/intercept, $\lambda$, optimization settings, selection rule, offset grid and chosen offset, and **one predeclared binary diagnostic slice** based on an admissible input. Use a threshold justified by training information or the scenario, not final-period errors. Include `confirmed_before_final: True` only once these choices are complete.

**For A1, do not refit on training plus validation after choosing the offset.** Keep the actual training-fitted model whose validation errors you used to choose the offset. This deliberately avoids a separate refitting/calibration problem. Also freeze the constant baseline: predict the training-target mean for every day.

**Your answer — 4.1 — plan**

Record the offset grid, tie rule, and predeclared slice before scoring offsets.

In [ ]:
# Implement planning cost; choose an offset on validation only.
# Create a fully specified, JSON-serializable frozen_record.
raise NotImplementedError("4.1: choose a decision rule and freeze all choices")

**Your answer — 4.1 — interpretation**

Write your answer here. Use your own results where requested.

### Final-period boundary — proceed only after freezing
The provided cell below loads the 84 later days from `data/final_period.csv`. Do not open or inspect this file before completing the frozen record. This is a **procedural holdout**, not an access-control mechanism.

“One final evaluation” means no changing features, hyperparameters, preprocessing, offsets, or slice definitions in response to these outcomes. You may rerun unchanged code to verify reproducibility. If you discover a genuine bug after inspection, disclose it and show the consequences; do not silently present a retuned result as untouched evaluation.

In [ ]:
assert (
    isinstance(frozen_record, dict)
    and frozen_record.get("confirmed_before_final") is True
), "Complete your frozen record before loading the final-period data."

final_period = pd.read_csv("data/final_period.csv", parse_dates=["date"])
assert len(final_period) == 84
assert final_period.date.min() > development.date.max()
assert set(final_period.record_id).isdisjoint(development.record_id)
print("Final period loaded:", final_period.date.min().date(), "to", final_period.date.max().date())

### 4.2 Evaluate without moving the goalposts (8 marks)
Reconstruct predictions **from the frozen record**, rather than a mutable variable from the sweep. Implement or use your own NumPy metric calculations. Produce:

**One overall table** for the constant baseline, selected point forecast, and selected reservation rule, with final-period RMSE, MAE, signed mean error (**prediction minus actual**), mean planning cost, and under-reservation frequency. Use nonnegative reservations for cost calculations. Label the reservation rule as a **decision**, not a new point predictor. Round displayed values sensibly; retain full precision for computations.

**One slice table** for the two groups declared before inspection, giving sample counts, point-forecast RMSE and signed mean error, and reservation cost. Do not drop a difficult group. An empty group should be reported as empty, not assigned a made-up zero error.

**One time-ordered residual plot** for the point forecast. Indicate the zero-error line and the sign convention. No additional tuning is allowed.

### 4.3 What did the holdout establish? (4 marks)
In about **150–200 words**, compare validation and final evidence. Include at least two actual numbers from your tables. Explain one reason that an aggregate metric may conceal operationally important error, and whether your results support deploying the point forecast, the reservation rule, both, or neither. Distinguish an observed later-period result from a guarantee about future periods.

In [ ]:
# Reconstruct solely from frozen_record; do not fit or tune anything here.
# Produce the overall table, prespecified slice table, and residual-over-time plot.
raise NotImplementedError("4.2: perform and interpret the frozen final evaluation")

**Your answer — 4.3**

Write your answer here. Use your own results where requested.

## Part 5 — Evidence memo and team accountability (8 marks)

### 5.1 Recommendation memo (6 marks)
Write **350–500 words** to the fictional operations manager. Make a recommendation and support it with your actual evidence. The memo should stand alone without claiming that a good score proves the model is correct. Include the prototype's main failure, an implementation-correctness check, an optimization-versus-generalization finding, the final prediction/decision results, a limitation, and a concrete monitoring or next-evidence requirement. Distinguish observed facts from hypotheses. Do not simply list all the experiments.

### 5.2 Contributions, revisions, and attribution (2 marks)
For each member, record a substantive contribution and a different part they reviewed. Give **one team belief or implementation choice that changed**, what evidence changed it, and the resulting action. Include sources/reused code and the required AI-use disclosure. “Everything was shared equally” without specifics is not a useful contribution record.

**Your answer — 5.1**

Write your answer here. Use your own results where requested.

**Your answer — 5.2**

Write your answer here. Use your own results where requested.

## Submission check

Before submitting, restart the kernel and run the completed notebook from top to bottom. Keep output tables and plots. Remove only irrelevant debugging clutter, not evidence of a meaningful failed experiment. The final notebook must not require network access or files other than the supplied `data/development.csv` and `data/final_period.csv`. Keep the supplied filenames and relative paths unchanged; do not modify the CSV files. Perform the documented Part 0.2 repairs in memory, preserving `development_raw`.

Check that the team roster, full answers, experiment settings, frozen record, final evaluation, source acknowledgements, and AI disclosure are present. Confirm that every member can explain the objective, the split, one correctness test, and the difference between the point forecast and the reservation rule.

## Lecture connections and scope

| Supplied course material | Used here |
|:--|:--|
| M01, *Models, People, and the Road Ahead*; M02, *Supervised, unsupervised, self- & semi-supervised* | Task, decision context, generalization, and the limit of clustering for this problem |
| M04, *Supervised learning via regression, risk, and loss* | Linear regression, empirical risk, vectorization, and an independent least-squares reference |
| M08, *Maximum likelihood estimation — principle & examples* | Gaussian likelihood, log-likelihood, and assumptions |
| M12, *Evaluation metrics for regression & classification*; M13, *Train/validation/test splits & cross-validation* | Regression metrics, decision-aligned evaluation, leakage, chronological evaluation, and slices |
| M14, *Bias-variance tradeoff & learning curves* | Interpreting learning curves without overclaiming a decomposition |
| M16, *Regularization — L2, L1, and Beyond*; M17, *Optimization basics — gradient descent & variants* | Ridge penalty, model selection, optimization diagnostics, and batch GD |

This assignment uses those foundations in a new, synthetic investigation. It does **not** require EM or the later material on SVMs, feature construction, dimensionality reduction, or neural networks. The mathematical normalization, data provenance, reservation cost, and experiment requirements above are specific to this assignment.